# Week 8 Day 2: Guided Practice — Modernize Your CNN

**Dataset:** Fashion-MNIST (same as Day 1)
**Time:** 20 minutes (instructor-led)
**Scaffolding:** ~70% provided

---

## Goal

Take **yesterday's CNN** and apply three ideas from this morning's lecture — and discover an honest tradeoff:
1. The **conv block** + **Batch Normalization** — a reusable `Conv → BN → ReLU → Pool` unit.
2. **Global Average Pooling** swapped in naively — watch the parameters *collapse*... and the accuracy *fall*. Why?
3. **GAP done right** — pair it with the lecture's *channels-up* rule and watch accuracy come back, with the giant Dense head still gone.

**The number to beat:** Day 1's CNN was **110,634 parameters**, and the `Flatten → Dense(64)` head alone was **101,066** of them (~91%).

## Part 1: Setup and Data (Provided)

In [1]:
# Setup (provided)
import os
os.environ['KERAS_BACKEND'] = 'torch'
import keras
from keras import layers
import numpy as np
keras.utils.set_random_seed(42)
print('Keras', keras.__version__, '| backend:', keras.backend.backend())

Keras 3.13.2 | backend: torch


In [2]:
# Load + preprocess Fashion-MNIST (provided — same as Day 1)
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = (X_train_full.astype('float32') / 255.0)[..., np.newaxis]   # add channel dim
X_test = (X_test.astype('float32') / 255.0)[..., np.newaxis]
X_train, X_val = X_train_full[:50000], X_train_full[50000:]
y_train, y_val = y_train_full[:50000], y_train_full[50000:]
print('train', X_train.shape, '| val', X_val.shape, '| test', X_test.shape)

EPOCHS = 14   # GAP nets converge a little slower; this lets the channels-up model catch up
def train_eval(model):
    """Compile, train, report params + test accuracy (provided helper)."""
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=EPOCHS, batch_size=128, verbose=2)
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f'\n>>> {model.name}: {model.count_params():,} params | test acc {acc:.4f}')
    return {'name': model.name, 'params': model.count_params(), 'test_acc': acc}

results = []

train (50000, 28, 28, 1) | val (10000, 28, 28, 1) | test (10000, 28, 28, 1)


## Part 2: Recap — Yesterday's CNN (Provided)

The Day-1 model, unchanged: `Conv → Pool → Conv → Pool → Flatten → Dense(64) → Dense(10)`.
Note that the head (`Flatten → Dense(64) → Dense(10)`) holds **101,066** of the 110,634 parameters.

In [3]:
# Day-1 baseline (provided)
keras.utils.set_random_seed(42)
model_day1 = keras.Sequential([
    layers.Input((28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),   # the head: ~91% of the model
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax'),
], name='day1_baseline')
results.append(train_eval(model_day1))

Epoch 1/14


391/391 - 7s - 19ms/step - accuracy: 0.7526 - loss: 0.6826 - val_accuracy: 0.8366 - val_loss: 0.4352


Epoch 2/14


391/391 - 7s - 18ms/step - accuracy: 0.8485 - loss: 0.4266 - val_accuracy: 0.8739 - val_loss: 0.3480


Epoch 3/14


...

Epoch 13/14


391/391 - 7s - 18ms/step - accuracy: 0.9194 - loss: 0.2178 - val_accuracy: 0.9087 - val_loss: 0.2432


Epoch 14/14


391/391 - 7s - 18ms/step - accuracy: 0.9219 - loss: 0.2129 - val_accuracy: 0.9070 - val_loss: 0.2483



>>> day1_baseline: 110,634 params | test acc 0.9026


## Part 3: Naive Global Average Pooling — the trap

**YOUR TASK (two blanks):**
1. Finish the `conv_block` by adding **BatchNormalization** (so the block is `Conv → BN → ReLU → Pool`).
2. Replace the whole `Flatten → Dense(64) → Dropout` head with a single **GlobalAveragePooling2D**.

Then predict, before you run: the parameters will *collapse*. What will happen to **accuracy**?

In [4]:
# TODO 1: add BatchNormalization inside the block
def conv_block(filters):
    return [
        layers.Conv2D(filters, 3, padding='same'),
        layers.BatchNormalization(),       # <-- ANSWER 1: the modern addition
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
    ]

# TODO 2: swap the Flatten+Dense head for GlobalAveragePooling2D
keras.utils.set_random_seed(42)
model_gap_naive = keras.Sequential([
    layers.Input((28, 28, 1)),
    *conv_block(32),      # the * unpacks the block's layers — reuse is the point of a block
    *conv_block(32),      # trunk stays narrow: only 32 channels reach the head
    layers.GlobalAveragePooling2D(),   # <-- ANSWER 2: replaces Flatten + Dense(64) + Dropout
    layers.Dense(10, activation='softmax'),
], name='gap_naive')
results.append(train_eval(model_gap_naive))

Epoch 1/14


391/391 - 9s - 22ms/step - accuracy: 0.6473 - loss: 1.2338 - val_accuracy: 0.2052 - val_loss: 2.2309


Epoch 2/14


391/391 - 8s - 21ms/step - accuracy: 0.7607 - loss: 0.7946 - val_accuracy: 0.6960 - val_loss: 0.8753


Epoch 3/14


...

Epoch 13/14


391/391 - 8s - 21ms/step - accuracy: 0.8636 - loss: 0.4015 - val_accuracy: 0.6843 - val_loss: 0.8675


Epoch 14/14


391/391 - 8s - 21ms/step - accuracy: 0.8684 - loss: 0.3905 - val_accuracy: 0.7676 - val_loss: 0.6202



>>> gap_naive: 10,154 params | test acc 0.7655


### PAUSE HERE — Discuss with Instructor

1. The model went from **110,634** parameters to about **10,000** — an ~11× collapse. Did accuracy *hold*?
2. It **dropped** (a lot). Why? GAP turns each feature map into a single number, so the classifier only sees **as many features as there are channels** — here just **32**. That's too thin.
3. So GAP isn't a free win on its own. What does the lecture's *channels-up* rule suggest we do?

## Part 4: GAP Done Right — channels up

**YOUR TASK:** give GAP enough to work with. Build a **channels-up** trunk — `32 → 64 → 128` — so the final block hands GAP **128** feature maps instead of 32. The head stays tiny; accuracy comes back.

In [5]:
# TODO: build a channels-up trunk (32 -> 64 -> 128) before the GAP head
keras.utils.set_random_seed(42)
model_gap_wide = keras.Sequential([
    layers.Input((28, 28, 1)),
    *conv_block(32),      # <-- ANSWER: channels grow
    *conv_block(64),      # <--          32 -> 64
    *conv_block(128),     # <--          64 -> 128  (now 128 features reach GAP)
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax'),
], name='gap_wide')
results.append(train_eval(model_gap_wide))

Epoch 1/14


391/391 - 9s - 24ms/step - accuracy: 0.8237 - loss: 0.5246 - val_accuracy: 0.5876 - val_loss: 1.1508


Epoch 2/14


391/391 - 9s - 24ms/step - accuracy: 0.8861 - loss: 0.3224 - val_accuracy: 0.8457 - val_loss: 0.4347


Epoch 3/14


...

Epoch 13/14


391/391 - 9s - 24ms/step - accuracy: 0.9553 - loss: 0.1306 - val_accuracy: 0.8347 - val_loss: 0.4562


Epoch 14/14


391/391 - 9s - 24ms/step - accuracy: 0.9607 - loss: 0.1178 - val_accuracy: 0.8870 - val_loss: 0.3222



>>> gap_wide: 94,858 params | test acc 0.8836


## Part 5: Compare — params, accuracy, and where the budget goes

In [6]:
# Comparison (provided)
HEAD_PARAMS = {'day1_baseline': 101_066, 'gap_naive': 330, 'gap_wide': 1_290}
print(f'{"Model":16s} | {"Params":>9s} | {"Head params":>11s} | {"Test Acc":>8s}')
print('-' * 56)
for r in results:
    print(f'{r["name"]:16s} | {r["params"]:>9,} | {HEAD_PARAMS[r["name"]]:>11,} | {r["test_acc"]:>8.4f}')

base = next(r for r in results if r['name'] == 'day1_baseline')
wide = next(r for r in results if r['name'] == 'gap_wide')
print(f'\nThe HEAD collapsed {HEAD_PARAMS["day1_baseline"]:,} -> {HEAD_PARAMS["gap_wide"]:,} '
      f'({HEAD_PARAMS["day1_baseline"]//HEAD_PARAMS["gap_wide"]}x).')
print(f'Total params stayed ~the same ({base["params"]:,} -> {wide["params"]:,}) — '
      f'we MOVED the budget out of the Dense head and INTO convolution.')

Model            |    Params | Head params | Test Acc
--------------------------------------------------------
day1_baseline    |   110,634 |     101,066 |   0.9026
gap_naive        |    10,154 |         330 |   0.7655
gap_wide         |    94,858 |       1,290 |   0.8836

The HEAD collapsed 101,066 -> 1,290 (78x).
Total params stayed ~the same (110,634 -> 94,858) — we MOVED the budget out of the Dense head and INTO convolution.


### The lesson (discuss)

- **GAP alone isn't free.** Swapping it in naively collapsed the parameters *but lost accuracy* — the classifier saw too few features.
- **GAP done right** keeps the head tiny (1,290 vs 101,066 — a 78× collapse) but **reinvests the freed budget into a deeper conv trunk** (channels up to 128). Total params stay ~the same as Day 1, and accuracy **recovers to ~88%** — within ~2 points of the Flatten baseline, versus the mid-70s for naive GAP. GAP trades a hair of accuracy *at this small scale* for a far more scalable design; on big networks it wins outright, because a Flatten head would be huge **and** overfit.
- That's the real modern pattern: **spend your budget on *seeing* (convolution), not on a fat Dense head.** It's exactly how ResNet and EfficientNet are built — and why GAP's payoff grows with scale, where a giant Dense head would also *overfit*.

## Bonus (if time): a skip connection

A 2-line taste of ResNet using the **functional API** — a block's output is added back to its input (`out = F(x) + x`). Optional, for the curious.

In [7]:
# Bonus (provided) — one residual block via the functional API
keras.utils.set_random_seed(42)
inp = layers.Input((28, 28, 1))
x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
shortcut = x
y = layers.Conv2D(32, 3, padding='same')(x); y = layers.BatchNormalization()(y); y = layers.Activation('relu')(y)
y = layers.Conv2D(32, 3, padding='same')(y); y = layers.BatchNormalization()(y)
y = layers.Add()([y, shortcut])          # <-- the skip connection: F(x) + x
y = layers.Activation('relu')(y)
y = layers.GlobalAveragePooling2D()(y)
out = layers.Dense(10, activation='softmax')(y)
model_resid = keras.Model(inp, out, name='with_skip')
print('residual model params:', f'{model_resid.count_params():,}')

residual model params: 19,402


## Wrap-up

You modernized Day-1's CNN and met an honest engineering tradeoff:
- **Conv block + BatchNorm** — a reusable, fast-training unit.
- **GAP naively** — collapses parameters but loses accuracy on a thin trunk.
- **GAP + channels-up** — collapses the *head* (78×) and reinvests the budget in convolution, holding accuracy.

These are the same building blocks inside the deep pretrained networks you'll **fine-tune next week** in Week 9.